# MIT-BIH Context-Aware ECG Beat Dataset Creator

## Objective
Create a context-aware ECG beat dataset using temporal beat windows for subject-independent training and evaluation.

## Key Features
1. **Context Windows**: 7 beats (3 previous + 1 center + 3 subsequent)
2. **Record-Wise Split**: No patient leakage between train/val/test
3. **R-Peak Detection**: Using Pan-Tompkins algorithm
4. **Fixed Beat Length**: 200 samples per beat (90 before R-peak, 110 after)
5. **Binary Classification**: N = Normal (0), any other = Abnormal (1)

## Constraints
- Skip record 119 entirely (reserved for live testing)
- Beat-wise random splitting is NOT allowed
- Context window strictly enforced

## STEP 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from glob import glob
from scipy.signal import resample, butter, filtfilt, find_peaks
import warnings
warnings.filterwarnings('ignore')

# Configuration
BEAT_LENGTH = 200  # Fixed samples per beat
PRE_R_SAMPLES = 90  # Samples before R-peak
POST_R_SAMPLES = 110  # Samples after R-peak
CONTEXT_WINDOW_SIZE = 7  # 3 previous + 1 center + 3 subsequent
SAMPLING_RATE = 360  # MIT-BIH sampling rate (Hz)
EXCLUDE_RECORDS = ['119']  # Reserve for live testing
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print('Configuration:')
print(f'  Beat length: {BEAT_LENGTH} samples')
print(f'  Pre-R samples: {PRE_R_SAMPLES}')
print(f'  Post-R samples: {POST_R_SAMPLES}')
print(f'  Context window size: {CONTEXT_WINDOW_SIZE} beats')
print(f'  Sampling rate: {SAMPLING_RATE} Hz')
print(f'  Excluded records: {EXCLUDE_RECORDS}')

## STEP 2: Define Data Paths

In [ ]:
# Try Kaggle path first, then local
KAGGLE_PATH = '/kaggle/input/mitbih-database/'
LOCAL_PATH = '../deploy/sample/'  # Fallback for local testing

if os.path.exists(KAGGLE_PATH):
    DATA_PATH = KAGGLE_PATH
    print(f'Using Kaggle path: {DATA_PATH}')
elif os.path.exists(LOCAL_PATH):
    DATA_PATH = LOCAL_PATH
    print(f'Using local path: {DATA_PATH}')
else:
    raise FileNotFoundError('No valid data path found!')

# List all CSV files (ECG signals)
signal_files = sorted(glob(os.path.join(DATA_PATH, '*.csv')))
print(f'\nFound {len(signal_files)} signal files')

# Filter out excluded records
signal_files = [f for f in signal_files if not any(excl in os.path.basename(f) for excl in EXCLUDE_RECORDS)]
print(f'After excluding {EXCLUDE_RECORDS}: {len(signal_files)} files')

# List available records
record_names = [os.path.basename(f).replace('.csv', '') for f in signal_files]
print(f'\nAvailable records: {record_names}')

## STEP 3: Parse Annotation File

In [ ]:
def parse_annotations(annotation_file):
    """
    Parse MIT-BIH annotation file.
    
    Returns:
        List of (sample_index, beat_type) tuples
    """
    annotations = []
    
    try:
        with open(annotation_file, 'r') as f:
            lines = f.readlines()
        
        # Skip header line
        for line in lines[1:]:
            parts = line.strip().split()
            if len(parts) >= 3:
                try:
                    sample_idx = int(parts[1])
                    beat_type = parts[2]
                    
                    # Skip non-beat annotations (rhythm markers, etc.)
                    # Valid beat types: N, L, R, B, A, a, J, S, V, r, F, e, j, n, E, /, f, Q, ?
                    if beat_type not in ['+', '~', '|', 's', 'T', '*', 'D', '=', '"', '@']:
                        annotations.append((sample_idx, beat_type))
                except (ValueError, IndexError):
                    continue
    except Exception as e:
        print(f'Error parsing {annotation_file}: {e}')
        
    return annotations


# Test parsing
test_ann_file = os.path.join(DATA_PATH, '100annotations.txt')
if os.path.exists(test_ann_file):
    test_annotations = parse_annotations(test_ann_file)
    print(f'Test parse: {len(test_annotations)} annotations from 100annotations.txt')
    print(f'First 5 annotations: {test_annotations[:5]}')

## STEP 4: R-Peak Detection (Pan-Tompkins Algorithm)

In [ ]:
def bandpass_filter(signal, lowcut=5.0, highcut=15.0, fs=360, order=1):
    """Apply bandpass filter to ECG signal."""
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


def detect_r_peaks_pan_tompkins(signal, fs=360):
    """
    Detect R-peaks using Pan-Tompkins algorithm.
    
    Steps:
    1. Bandpass filter (5-15 Hz)
    2. Differentiate
    3. Square
    4. Moving window integration
    5. Adaptive thresholding
    
    Returns:
        Array of R-peak indices
    """
    # Step 1: Bandpass filter
    filtered = bandpass_filter(signal, 5.0, 15.0, fs)
    
    # Step 2: Differentiate
    diff = np.diff(filtered)
    
    # Step 3: Square
    squared = diff ** 2
    
    # Step 4: Moving window integration (150ms window)
    window_size = int(0.150 * fs)
    integrated = np.convolve(squared, np.ones(window_size) / window_size, mode='same')
    
    # Step 5: Find peaks with adaptive thresholding
    # Use minimum distance between peaks (200ms = 0.2s * fs samples)
    min_distance = int(0.2 * fs)
    
    # Dynamic threshold: 0.5 * max of rolling mean
    threshold = 0.3 * np.max(integrated)
    
    peaks, _ = find_peaks(integrated, height=threshold, distance=min_distance)
    
    # Refine peaks: find actual R-peak in original signal (max in small window)
    refined_peaks = []
    search_window = int(0.05 * fs)  # 50ms window
    
    for peak in peaks:
        start = max(0, peak - search_window)
        end = min(len(signal), peak + search_window)
        local_max = start + np.argmax(signal[start:end])
        refined_peaks.append(local_max)
    
    return np.array(refined_peaks)


print('R-peak detection functions defined (Pan-Tompkins algorithm)')

## STEP 5: Beat Extraction Functions

In [ ]:
def extract_beat(signal, r_peak_idx, pre_samples=PRE_R_SAMPLES, post_samples=POST_R_SAMPLES):
    """
    Extract a fixed-length beat centered on R-peak.
    
    Args:
        signal: Full ECG signal array
        r_peak_idx: Index of R-peak
        pre_samples: Samples before R-peak (default: 90)
        post_samples: Samples after R-peak (default: 110)
        
    Returns:
        Beat array of length (pre_samples + post_samples)
    """
    beat_length = pre_samples + post_samples
    start_idx = r_peak_idx - pre_samples
    end_idx = r_peak_idx + post_samples
    
    # Handle edge cases with zero padding
    if start_idx < 0:
        pad_before = -start_idx
        beat = np.zeros(beat_length, dtype=np.float32)
        available = signal[:end_idx]
        beat[pad_before:pad_before + len(available)] = available
    elif end_idx > len(signal):
        beat = np.zeros(beat_length, dtype=np.float32)
        available = signal[start_idx:]
        beat[:len(available)] = available
    else:
        beat = signal[start_idx:end_idx].astype(np.float32)
    
    return beat


def find_annotation_for_peak(r_peak_idx, annotations, tolerance=20):
    """
    Find the annotation closest to the detected R-peak.
    
    Args:
        r_peak_idx: Detected R-peak index
        annotations: List of (sample_idx, beat_type) tuples
        tolerance: Maximum distance to match (samples)
        
    Returns:
        beat_type if found, None otherwise
    """
    best_match = None
    best_distance = tolerance + 1
    
    for sample_idx, beat_type in annotations:
        distance = abs(sample_idx - r_peak_idx)
        if distance < best_distance:
            best_distance = distance
            best_match = beat_type
    
    return best_match if best_distance <= tolerance else None


print('Beat extraction functions defined')

## STEP 6: Process Single Record

In [ ]:
def process_record(signal_file, annotation_file, use_detected_peaks=False):
    """
    Process a single MIT-BIH record to extract beats with labels.
    
    Args:
        signal_file: Path to signal CSV
        annotation_file: Path to annotation file
        use_detected_peaks: If True, use Pan-Tompkins detection; else use annotation peaks
        
    Returns:
        List of (beat, label, beat_type) tuples in temporal order
    """
    # Load signal (use MLII lead)
    try:
        signal_df = pd.read_csv(signal_file)
        signal_df.columns = signal_df.columns.str.strip().str.strip("'")
        
        if 'MLII' in signal_df.columns:
            signal = signal_df['MLII'].values.astype(float)
        elif signal_df.shape[1] > 1:
            signal = signal_df.iloc[:, 1].values.astype(float)
        else:
            signal = signal_df.iloc[:, 0].values.astype(float)
    except Exception as e:
        print(f'Error loading {signal_file}: {e}')
        return []
    
    # Parse annotations
    annotations = parse_annotations(annotation_file)
    if len(annotations) < 3:
        print(f'Too few annotations in {annotation_file}')
        return []
    
    # Use annotation R-peaks (more accurate than detection)
    r_peaks = np.array([ann[0] for ann in annotations])
    
    beats_data = []
    
    for i, r_peak_idx in enumerate(r_peaks):
        beat = extract_beat(signal, r_peak_idx)
        
        if beat is None or len(beat) != BEAT_LENGTH:
            continue
        
        # Get beat type from annotation
        beat_type = annotations[i][1]
        
        # Binary label: N = Normal (0), everything else = Abnormal (1)
        label = 0 if beat_type == 'N' else 1
        
        beats_data.append((beat, label, beat_type))
    
    return beats_data


print('Record processing function defined')

## STEP 7: Context Window Construction

In [ ]:
def create_context_windows(beats_data, window_size=CONTEXT_WINDOW_SIZE):
    """
    Create context windows from sequential beats.
    
    Context window size: 7 beats
    - 3 previous beats
    - 1 center beat (for classification)
    - 3 subsequent beats
    
    Args:
        beats_data: List of (beat, label, beat_type) tuples in temporal order
        window_size: Number of beats in context window
        
    Returns:
        List of (context_window, center_label, center_beat_type) tuples
        context_window shape: (window_size, beat_length)
    """
    if len(beats_data) < window_size:
        return []
    
    half_window = window_size // 2  # 3 beats before/after center
    context_windows = []
    
    for center_idx in range(half_window, len(beats_data) - half_window):
        # Extract beats in window
        window_beats = []
        for i in range(center_idx - half_window, center_idx + half_window + 1):
            beat, _, _ = beats_data[i]
            window_beats.append(beat)
        
        # Stack beats: shape (7, 200)
        context_window = np.stack(window_beats, axis=0)
        
        # Use center beat's label
        _, center_label, center_beat_type = beats_data[center_idx]
        
        context_windows.append((context_window, center_label, center_beat_type))
    
    return context_windows


print(f'Context window function defined (window_size={CONTEXT_WINDOW_SIZE})')

## STEP 8: Process All Records

In [ ]:
# Process all records and create context windows
all_record_data = {}  # record_id -> list of context windows
record_stats = []

print('Processing MIT-BIH records...')
print('=' * 70)

for signal_file in signal_files:
    record_name = os.path.basename(signal_file).replace('.csv', '')
    
    # Skip excluded records
    if record_name in EXCLUDE_RECORDS:
        print(f'Skipping {record_name} (reserved for testing)')
        continue
    
    # Find annotation file
    annotation_file = os.path.join(DATA_PATH, f'{record_name}annotations.txt')
    if not os.path.exists(annotation_file):
        # Try alternative naming
        annotation_file = os.path.join(DATA_PATH, f'a{record_name}annotations.txt')
        if not os.path.exists(annotation_file):
            print(f'No annotation file for {record_name}, skipping')
            continue
    
    # Process record to get individual beats
    beats_data = process_record(signal_file, annotation_file)
    
    if len(beats_data) < CONTEXT_WINDOW_SIZE:
        print(f'Record {record_name}: Too few beats ({len(beats_data)}), skipping')
        continue
    
    # Create context windows
    context_windows = create_context_windows(beats_data)
    
    if len(context_windows) > 0:
        n_normal = sum(1 for _, label, _ in context_windows if label == 0)
        n_abnormal = len(context_windows) - n_normal
        
        print(f'Record {record_name}: {len(context_windows)} context windows '
              f'(Normal: {n_normal}, Abnormal: {n_abnormal})')
        
        all_record_data[record_name] = context_windows
        record_stats.append({
            'record': record_name,
            'total_windows': len(context_windows),
            'normal': n_normal,
            'abnormal': n_abnormal
        })

print('\n' + '=' * 70)
print(f'Processed {len(all_record_data)} records')
total_windows = sum(len(cw) for cw in all_record_data.values())
print(f'Total context windows: {total_windows}')

## STEP 9: Record-Wise Train/Val/Test Split

In [ ]:
# Split records (not beats!) into train/val/test
# This ensures no patient leakage

all_records = list(all_record_data.keys())
np.random.shuffle(all_records)

n_records = len(all_records)
n_train = int(0.7 * n_records)
n_val = int(0.15 * n_records)
n_test = n_records - n_train - n_val

train_records = all_records[:n_train]
val_records = all_records[n_train:n_train + n_val]
test_records = all_records[n_train + n_val:]

print('Record-Wise Split (No Patient Leakage):')
print('=' * 50)
print(f'Train records ({len(train_records)}): {train_records}')
print(f'Val records ({len(val_records)}): {val_records}')
print(f'Test records ({len(test_records)}): {test_records}')

# Verify no overlap
assert len(set(train_records) & set(val_records)) == 0, 'Train/Val overlap!'
assert len(set(train_records) & set(test_records)) == 0, 'Train/Test overlap!'
assert len(set(val_records) & set(test_records)) == 0, 'Val/Test overlap!'
print('\n✓ Verified: No overlap between splits')

## STEP 10: Collect Data for Each Split

In [ ]:
def collect_split_data(record_list, all_data):
    """
    Collect all context windows and labels from specified records.
    
    Returns:
        X: numpy array of shape (n_samples, context_window_size, beat_length)
        y: numpy array of shape (n_samples,)
    """
    X_list = []
    y_list = []
    
    for record in record_list:
        if record in all_data:
            for context_window, label, _ in all_data[record]:
                X_list.append(context_window)
                y_list.append(label)
    
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    
    return X, y


# Collect data for each split
X_train, y_train = collect_split_data(train_records, all_record_data)
X_val, y_val = collect_split_data(val_records, all_record_data)
X_test, y_test = collect_split_data(test_records, all_record_data)

print('Data Collected:')
print('=' * 50)
print(f'Train: X={X_train.shape}, y={y_train.shape}')
print(f'  Normal: {np.sum(y_train == 0)}, Abnormal: {np.sum(y_train == 1)}')
print(f'Val: X={X_val.shape}, y={y_val.shape}')
print(f'  Normal: {np.sum(y_val == 0)}, Abnormal: {np.sum(y_val == 1)}')
print(f'Test: X={X_test.shape}, y={y_test.shape}')
print(f'  Normal: {np.sum(y_test == 0)}, Abnormal: {np.sum(y_test == 1)}')

## STEP 11: Data Quality Checks

In [ ]:
print('Data Quality Checks')
print('=' * 50)

# Check for NaN values
train_nan = np.isnan(X_train).sum()
val_nan = np.isnan(X_val).sum()
test_nan = np.isnan(X_test).sum()
print(f'NaN values - Train: {train_nan}, Val: {val_nan}, Test: {test_nan}')

# Check for infinite values
train_inf = np.isinf(X_train).sum()
val_inf = np.isinf(X_val).sum()
test_inf = np.isinf(X_test).sum()
print(f'Infinite values - Train: {train_inf}, Val: {val_inf}, Test: {test_inf}')

# Check data ranges
print(f'\nTrain data range: [{X_train.min():.2f}, {X_train.max():.2f}]')
print(f'Val data range: [{X_val.min():.2f}, {X_val.max():.2f}]')
print(f'Test data range: [{X_test.min():.2f}, {X_test.max():.2f}]')

# Check shape
expected_shape = (CONTEXT_WINDOW_SIZE, BEAT_LENGTH)
print(f'\nExpected sample shape: {expected_shape}')
print(f'Actual sample shape: {X_train[0].shape}')
assert X_train[0].shape == expected_shape, 'Incorrect sample shape!'
print('✓ Sample shape verified')

## STEP 12: Visualize Sample Context Windows

In [ ]:
import matplotlib.pyplot as plt

# Sample context windows
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

# Get sample normal and abnormal windows
normal_idx = np.where(y_train == 0)[0][:4]
abnormal_idx = np.where(y_train == 1)[0][:4]

for i, idx in enumerate(normal_idx):
    if i >= 4:
        break
    ax = axes[0, i]
    context = X_train[idx]
    for beat_idx in range(CONTEXT_WINDOW_SIZE):
        alpha = 0.3 if beat_idx != 3 else 1.0  # Highlight center beat
        color = 'blue' if beat_idx == 3 else 'gray'
        ax.plot(context[beat_idx], alpha=alpha, color=color, linewidth=1.5 if beat_idx == 3 else 0.8)
    ax.set_title(f'Normal (Center Beat)')
    ax.set_xlabel('Sample')
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

for i, idx in enumerate(abnormal_idx):
    if i >= 4:
        break
    ax = axes[1, i]
    context = X_train[idx]
    for beat_idx in range(CONTEXT_WINDOW_SIZE):
        alpha = 0.3 if beat_idx != 3 else 1.0
        color = 'red' if beat_idx == 3 else 'gray'
        ax.plot(context[beat_idx], alpha=alpha, color=color, linewidth=1.5 if beat_idx == 3 else 0.8)
    ax.set_title(f'Abnormal (Center Beat)')
    ax.set_xlabel('Sample')
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

plt.suptitle('Context Windows (7 Beats - Center Beat Highlighted)', fontsize=14)
plt.tight_layout()
plt.savefig('context_windows_visualization.png', dpi=150)
plt.show()

print('Visualization saved to: context_windows_visualization.png')

## STEP 13: Save Output Files

In [ ]:
# Output directory
output_dir = '/kaggle/working/' if os.path.exists('/kaggle/working/') else './'

# Save numpy arrays
np.save(os.path.join(output_dir, 'X_train.npy'), X_train)
np.save(os.path.join(output_dir, 'y_train.npy'), y_train)
np.save(os.path.join(output_dir, 'X_val.npy'), X_val)
np.save(os.path.join(output_dir, 'y_val.npy'), y_val)
np.save(os.path.join(output_dir, 'X_test.npy'), X_test)
np.save(os.path.join(output_dir, 'y_test.npy'), y_test)

print('Saved numpy arrays:')
print(f'  X_train.npy: {X_train.shape}')
print(f'  y_train.npy: {y_train.shape}')
print(f'  X_val.npy: {X_val.shape}')
print(f'  y_val.npy: {y_val.shape}')
print(f'  X_test.npy: {X_test.shape}')
print(f'  y_test.npy: {y_test.shape}')

# Save record split
record_split = {
    'train': train_records,
    'val': val_records,
    'test': test_records
}
with open(os.path.join(output_dir, 'record_split.json'), 'w') as f:
    json.dump(record_split, f, indent=2)
print('\nSaved record_split.json')

# Save dataset config
dataset_config = {
    'beat_length': BEAT_LENGTH,
    'pre_r_samples': PRE_R_SAMPLES,
    'post_r_samples': POST_R_SAMPLES,
    'context_window_size': CONTEXT_WINDOW_SIZE,
    'sampling_rate': SAMPLING_RATE,
    'excluded_records': EXCLUDE_RECORDS,
    'label_mapping': {
        '0': 'Normal (N)',
        '1': 'Abnormal (non-N)'
    },
    'train_samples': int(len(X_train)),
    'val_samples': int(len(X_val)),
    'test_samples': int(len(X_test)),
    'train_normal': int(np.sum(y_train == 0)),
    'train_abnormal': int(np.sum(y_train == 1)),
    'val_normal': int(np.sum(y_val == 0)),
    'val_abnormal': int(np.sum(y_val == 1)),
    'test_normal': int(np.sum(y_test == 0)),
    'test_abnormal': int(np.sum(y_test == 1)),
    'random_state': RANDOM_STATE
}
with open(os.path.join(output_dir, 'dataset_config.json'), 'w') as f:
    json.dump(dataset_config, f, indent=2)
print('Saved dataset_config.json')

## STEP 14: Summary

In [ ]:
print('=' * 70)
print('CONTEXT-AWARE DATASET CREATION SUMMARY')
print('=' * 70)

print(f'\n1. INPUT:')
print(f'   - Source: MIT-BIH Arrhythmia Database')
print(f'   - Records processed: {len(all_record_data)}')
print(f'   - Records excluded: {EXCLUDE_RECORDS}')

print(f'\n2. CONTEXT WINDOW CONFIGURATION:')
print(f'   - Window size: {CONTEXT_WINDOW_SIZE} beats')
print(f'   - Structure: 3 previous + 1 center + 3 subsequent')
print(f'   - Beat length: {BEAT_LENGTH} samples')
print(f'   - Pre-R samples: {PRE_R_SAMPLES}')
print(f'   - Post-R samples: {POST_R_SAMPLES}')

print(f'\n3. RECORD-WISE SPLIT (No Patient Leakage):')
print(f'   - Train: {len(train_records)} records ({len(X_train)} windows)')
print(f'   - Val: {len(val_records)} records ({len(X_val)} windows)')
print(f'   - Test: {len(test_records)} records ({len(X_test)} windows)')

print(f'\n4. CLASS DISTRIBUTION:')
print(f'   Train: Normal={np.sum(y_train == 0)}, Abnormal={np.sum(y_train == 1)}')
print(f'   Val: Normal={np.sum(y_val == 0)}, Abnormal={np.sum(y_val == 1)}')
print(f'   Test: Normal={np.sum(y_test == 0)}, Abnormal={np.sum(y_test == 1)}')

print(f'\n5. OUTPUT FILES:')
print(f'   - X_train.npy, y_train.npy')
print(f'   - X_val.npy, y_val.npy')
print(f'   - X_test.npy, y_test.npy')
print(f'   - record_split.json')
print(f'   - dataset_config.json')

print(f'\n6. VALIDATION:')
print(f'   ✓ No patient leakage (record-wise split)')
print(f'   ✓ Context window strictly enforced ({CONTEXT_WINDOW_SIZE} beats)')
print(f'   ✓ Binary classification (Normal vs Abnormal)')
print(f'   ✓ Record 119 excluded (for live testing)')

print(f'\n7. NEXT STEPS:')
print(f'   - Run mitbih_context_cnn1d_training.ipynb')
print(f'   - Use output .npy files as input')
print(f'   - Normalize using training data statistics ONLY')

print('\n' + '=' * 70)
print('Dataset creation complete!')
print('=' * 70)